In [7]:
import torch, torchvision
import torch.nn.functional as F

MODEL_PATH = 'best_model_resnet18.pth'
CLASS_NAMES = ['Left', 'NoSign', 'Right']

model = torchvision.models.resnet18(pretrained=False)
model.fc = torch.nn.Linear(512, 3)
state = torch.load(MODEL_PATH, map_location='cpu')
model.load_state_dict(state)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device).eval().half()

import torchvision.transforms as T
import PIL.Image
import numpy as np

mean = torch.tensor([0.485, 0.456, 0.406], device=device).half()
std  = torch.tensor([0.229, 0.224, 0.225], device=device).half()

def preprocess(image_np):
    img = PIL.Image.fromarray(image_np)
    x = T.functional.to_tensor(img).to(device).half()
    x.sub_(mean[:, None, None]).div_(std[:, None, None])
    return x.unsqueeze(0)

import traitlets
import ipywidgets as widgets
from IPython.display import display
from jetbot import Camera, bgr8_to_jpeg

camera = Camera.instance(width=224, height=224)
image = widgets.Image(format='jpeg', width=224, height=224)
pred_label = widgets.HTML("<h2 style='text-align: center;'>Waiting...</h2>")

ui = widgets.VBox([image, pred_label])
camera_link = traitlets.dlink((camera, 'value'), (image, 'value'), transform=bgr8_to_jpeg)
display(ui)

import time

def update(change):
    frame = change['new']
    x = preprocess(frame)
    
    with torch.no_grad():
        y = model(x)
        p = F.softmax(y, dim=1).squeeze().float().cpu().numpy()
    
    # Apply thresholds
    left_prob = p[0]
    nosign_prob = p[1]
    right_prob = p[2]
    
    if left_prob > 0.9:
        result = "Left"
        color = "blue"
    elif right_prob > 0.9:
        result = "Right"
        color = "green"
    elif nosign_prob > 0.7:
        result = "No Sign"
        color = "gray"
    else:
        result = "Uncertain"
        color = "orange"
    
    pred_label.value = f"<h2 style='text-align: center; color: {color};'>{result}</h2>"
    time.sleep(0.001)

update({'new': camera.value})
camera.observe(update, names='value')

In [6]:
camera.unobserve(update, names='value')
time.sleep(0.1)
camera_link.unlink()
camera.stop()